In [1]:
import pandas as pd
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
export_dir = os.getcwd()
from pathlib import Path
import pickle

from collections import defaultdict
import time
import torch
import torch.nn as nn
import copy
import torch.nn.functional as F
import optuna
import logging
import matplotlib.pyplot as plt
import ipynb
import importlib
import sys
from pathlib import Path
from collections import Counter

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split 
from recbole.data import create_dataset, data_preparation
from recbole.config import Config



2026-04-17 14:55:26.983658: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 14:55:27.093455: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-17 14:55:27.093496: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-17 14:55:27.105507: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-17 14:55:27.138920: I tensorflow/core/platform/cpu_feature_guar

In [2]:
data_name = "LASTFM" ### Can be ML1M, lastfm
recommender_name = "VAE" ## Can be MLP, VAE, MLP_model, GMF_model, NCF
DP_DIR = Path("data/", data_name) 
export_dir = Path(os.getcwd())
files_path = Path(export_dir, DP_DIR)
checkpoints_path = Path(export_dir, "checkpoints")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
train_data = pd.read_csv(Path(files_path,f'Train_Data_{data_name}_New.csv'), index_col=0)

test_data = pd.read_csv(Path(files_path,f'Test_Data_{data_name}_New.csv'), index_col=0)

static_test_data = pd.read_csv(Path(files_path,f'Static_Test_Data_{data_name}_New.csv'), index_col=0)


Itesm_df = pd.read_csv(Path(files_path,f'Items_Data_{data_name}_New.csv'), index_col=0)


with open(Path(files_path,f'pop_dict_{data_name}.pkl'), 'rb') as f:
    pop_dict = pickle.load(f)

    
train_array = train_data.to_numpy()
test_array = test_data.to_numpy()

num_items=len(train_data.iloc[:,:-1].columns)
num_users=np.concatenate([train_array,test_array],axis=0).shape[0]

items_array = np.eye(num_items)
all_items_tensor = torch.Tensor(items_array).to(device)
all_array=np.concatenate((train_array, test_array))

In [4]:
df_inter = pd.read_csv('/home/mvarasteh/ProvidersExplantion/data/LASTFM/user_data.csv')
df_items = pd.read_csv('/home/mvarasteh/ProvidersExplantion/data/LASTFM/music_data_1.csv')
## removing nan values
df_inter=df_inter.dropna()
df_items=df_items.dropna()

In [5]:
## Capturing track IDS for .inter and .items files
items_in_items=set(df_items['track_id'])
items_in_inter=set(df_inter['track_id'])

## Finding mutual items in both files (output is ids of items)
mutual_items=list(items_in_items & items_in_inter)

In [6]:
df_inter=df_inter.set_index('track_id').loc[mutual_items]
df_inter=df_inter.reset_index()

## Remvoing users and items with less than k interactions

In [7]:
def filter_k_core(df, k):
    prev_len = 0
    while len(df) != prev_len:
        prev_len = len(df)
        # Filter items
        df = df[df.groupby('track_id')['user_id'].transform('count') > k]
        # Filter users
        df = df[df.groupby('user_id')['track_id'].transform('count') > k]
    return df

df_inter_filt = filter_k_core(df_inter, k=30)

# Filtering df_inter_items

In [8]:
## Capturing track IDS for .inter and .items files
items_in_items=set(df_items['track_id'])
items_in_inter=set(df_inter_filt['track_id'])

## Finding mutual items in both files (output is ids of items)
mutual_items=list(items_in_inter.intersection(items_in_items))
#mutual_items=list(items_in_items & items_in_inter)


df_items=df_items.set_index('track_id').loc[mutual_items]
df_items=df_items.reset_index()

df_inter=df_inter_filt

# Label Encoder for users and items id

In [9]:
from sklearn.preprocessing import LabelEncoder

user_encoder=LabelEncoder()
track_encoder=LabelEncoder()

df_inter['user_id_new']=user_encoder.fit_transform(df_inter['user_id'])

df_inter['track_id_new']=track_encoder.fit_transform(df_inter['track_id'])

df_items['track_id_new']=track_encoder.transform(df_items['track_id'])

In [10]:
# removing the clolumns with the previous id
df_items=df_items.drop(columns=['track_id'])
# renaming the teack_id column
df_items=df_items.rename(columns={'track_id_new': 'track_id'})

In [ ]:
df_items.to_csv('/home/mvarasteh/ProvidersExplantion/data/LASTFM/Items_Data_LASTFM_New.csv')

In [11]:
df_items.rename(columns={'track_id': 'item_id'}, inplace=True)

In [12]:
df_inter=df_inter.drop(columns=['track_id', 'user_id'])
df_inter=df_inter.rename(columns={'user_id_new':'user_id','track_id_new':'track_id' })

In [ ]:
df_inter.to_csv('/home/mvarasteh/ProvidersExplantion/data/LASTFM/user_data_playcount.csv')

In [13]:
playcounts=df_inter.groupby('track_id')['playcount'].sum().to_dict()


In [ ]:
with open('/home/mvarasteh/ProvidersExplantion/data/LASTFM/playcounts.pkl', 'wb') as f:
   pickle.dump(playcounts, f)

In [14]:
from scipy.sparse import csr_matrix

user_indices=df_inter['user_id'].values
items_indices=df_inter['track_id'].values
interactions=np.ones(len(user_indices))
input_array=csr_matrix((interactions,(user_indices,items_indices ) ))

In [15]:


df_sparse = pd.DataFrame(input_array.toarray(), dtype=int)

df_sparse['user_ids']=df_sparse.index

In [16]:
train_df, test_df = train_test_split(df_sparse, train_size=0.8, test_size=0.2, random_state=44, shuffle=False)

In [ ]:
train_df.to_csv('/home/mvarasteh/ProvidersExplantion/data/lastfm/Train_Data_LASTFM_New.csv')

test_df.to_csv('/home/mvarasteh/ProvidersExplantion/data/lastfm/Test_Data_LASTFM_New.csv')

In [17]:
input_array=all_array

## Popularity of items

In [18]:
pop_score=(input_array.sum(axis=0) / input_array.sum(axis=0).max())
pop_score=np.asarray(pop_score).reshape(-1)
pop_dict=dict(zip(np.arange(input_array.shape[1]), pop_score))

In [19]:
pop_array = np.zeros(len(pop_dict))
for key, value in pop_dict.items():
    pop_array[key] = value

In [20]:
num_items=input_array.shape[1]

In [21]:
kw_dict={'num_items':num_items, 'pop_array':pop_array}

In [22]:
# a function that samples different train data variation for a diverse training
def sample_indices(data, **kw):
    num_items = kw['num_items']
    pop_array = kw['pop_array']
    
    matrix = np.array(data)[:,:num_items] # keep only items columns, remove demographic features columns
    zero_indices = []
    one_indices = []
    for row in matrix:
        zero_idx = np.where(row == 0)[0]
        one_idx = np.where(row == 1)[0]
        probs = pop_array[zero_idx]
        probs = probs/ np.sum(probs)

        sampled_zero = np.random.choice(zero_idx, p = probs) # sample negative interactions according to items popularity 
        zero_indices.append(sampled_zero)
        
        sampled_one = np.random.choice(one_idx) # sample positive interactions from user's history
        data.iloc[row, sampled_one] = 0
        
        one_indices.append(sampled_one)
    data['pos'] = one_indices
    data['neg'] = zero_indices
    return data

In [29]:
static_test_data=sample_indices(test_df.drop(columns=['user_ids']).copy(), **kw_dict)

In [ ]:
static_test_data=static_test_data.drop(columns=['user_ids'])

In [ ]:

static_test_data.to_csv('/home/mvarasteh/ProvidersExplantion/data/lastfm/Static_Test_Data_LASTFM_New.csv')

In [30]:
train_data=train_df
test_data=test_df
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
train_data=train_data.iloc[:,:-1]
test_data=test_data.iloc[:,:-1]
#static_test_data=static_test_data.iloc[:,:-1]



train_array = train_data.to_numpy()
test_array = test_data.to_numpy()


num_items=len(train_data.columns)
num_users=len(train_data.index)+len(test_data.index)



items_array = np.eye(num_items)
all_items_tensor = torch.Tensor(items_array).to(device)
all_array=np.concatenate((train_array, test_array))

In [40]:
from ipynb.fs.defs.recommenders_architecture import *
importlib.reload(ipynb.fs.defs.recommenders_architecture)
from ipynb.fs.defs.recommenders_architecture import *

In [41]:
from ipynb.fs.defs.help_functions import *
importlib.reload(ipynb.fs.defs.help_functions)
from ipynb.fs.defs.help_functions import *

In [50]:
output_type_dict = {
    "VAE":"multiple",
    "MLP":"single"
}
output_type = output_type_dict[recommender_name] ### Can be single, multiple

In [51]:
num_items=all_array.shape[1]
kw_dict = {'device':device,
          'num_items': num_items,
          'pop_array':pop_array,
          'all_items_tensor':all_items_tensor,
          'static_test_data':static_test_data,
          'items_array':items_array,
          'output_type':output_type,
          'recommender_name':'MLP'}

In [52]:
output_type_dict = {
    "VAE":"multiple",
    "MLP":"single"
}


recommender_path_dict = {
    ("ML1M","VAE"): Path(checkpoints_path, "VAE_ML1M_17.pt"),

    ("lastfm","MLP"): Path(checkpoints_path, "MLP_lastfm_0.0001_512_0_19.pt"),
    ("ML1M","MLP"):Path(checkpoints_path, "MLP_ML1M_0.0_512_0_42.pt"),
    
    ("Yahoo","VAE"): Path(checkpoints_path, "VAE_Yahoo_0.0001_128_13.pt"),
    ("Yahoo","MLP"):Path(checkpoints_path, "MLP2_Yahoo_0.0083_128_1.pt"),

    ("Pinterest","VAE"): Path(checkpoints_path, "VAE_Pinterest_0.0002_32_12.pt"),
    ("Pinterest","MLP"):Path(checkpoints_path, "MLP_Pinterest_0.0062_512_21_0.pt")
    
}

hidden_dim_dict = {
    ("ML1M","VAE"): None,
    ("ML1M","MLP"): 128,

    ("Yahoo","VAE"): None,
    ("Yahoo","MLP"):32,
    
    ("Pinterest","VAE"): None,
    ("Pinterest","MLP"):512,
    ("lastfm", "MLP"): 128

}
data_name="lastfm"
output_type = output_type_dict[recommender_name] ### Can be single, multiple
hidden_dim = hidden_dim_dict[(data_name,recommender_name)]
recommender_path = recommender_path_dict[(data_name,recommender_name)]

KeyError: ('lastfm', 'VAE')

In [53]:
train_losses_dict = {}
test_losses_dict = {}
HR10_dict = {}

def MLP_objective(trial):
    
    #lr = trial.suggest_float('learning_rate', 0.001, 0.01)
    lr=0.001
    #batch_size = trial.suggest_categorical('batch_size', [256, 512, 1024])
    batch_size=512
    #hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128, 256, 512])
    hidden_dim=128
    beta=0.8670671663707932
    #beta = trial.suggest_float('beta', 0, 4) # hyperparameter that weights the different loss terms
    epochs = 30
    model = MLP(hidden_dim, **kw_dict)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    hr10 = []
    
    print(f'======================== new run - {recommender_name} ========================')
    logger.info(f'======================== new run - {recommender_name} ========================')
    
    num_training = train_data.shape[0]
    num_batches = int(np.ceil(num_training / batch_size))

    
    for epoch in range(epochs):
        train_matrix = sample_indices_(train_data.copy(), **kw_dict)
        #print(f"train_matrix shape{train_matrix.shape}")
        perm = np.random.permutation(num_training)
        loss = []
        train_pos_loss=[]
        train_neg_loss=[]
        if epoch!=0 and epoch%10 == 0: # decrease the learning rate every 10 epochs
            lr = 0.1*lr
            optimizer.lr = lr
        
        for b in range(num_batches):
            optimizer.zero_grad()
            if (b + 1) * batch_size >= num_training:
                batch_idx = perm[b * batch_size:]
            else:
                batch_idx = perm[b * batch_size: (b + 1) * batch_size]    
            batch_matrix = torch.FloatTensor(train_matrix[batch_idx,:-2]).to(device)
            batch_pos_idx = train_matrix[batch_idx,-2].astype(int)
            batch_neg_idx = train_matrix[batch_idx,-1].astype(int)

            batch_pos_items = torch.Tensor(items_array[batch_pos_idx]).to(device)
            batch_neg_items = torch.Tensor(items_array[batch_neg_idx]).to(device)

            #print(f"batch_matrix {batch_matrix.shape} and batch_pos_items {batch_pos_items.shape} ")

            pos_output = torch.diagonal(model(batch_matrix, batch_pos_items))
            #print('done')
            neg_output = torch.diagonal(model(batch_matrix, batch_neg_items))
            
            # MSE loss
            pos_loss = torch.mean((torch.ones_like(pos_output)-pos_output)**2)
            neg_loss = torch.mean((neg_output)**2)
            
            batch_loss = pos_loss + beta*neg_loss
            batch_loss.backward()
            optimizer.step()
            
            loss.append(batch_loss.item())
            train_pos_loss.append(pos_loss.item())
            train_neg_loss.append(neg_loss.item())
            
        print(f'train pos_loss = {np.mean(train_pos_loss)}, neg_loss = {np.mean(train_neg_loss)}')    
        train_losses.append(np.mean(loss))
        checkpoints_path='/home/mvarasteh/ProvidersExplantion/checkpoints/'
        torch.save(model.state_dict(), Path(checkpoints_path, f'MLP_lastfm_{round(lr,4)}_{batch_size}_{trial.number}_{epoch}.pt'))

        model.eval()
        test_matrix = np.array(static_test_data)
        test_tensor = torch.Tensor(test_matrix[:,:-2]).to(device)
        
        test_pos = test_matrix[:,-2].astype(int)
        test_neg = test_matrix[:,-1].astype(int)
        
        row_indices = np.arange(test_matrix.shape[0])
        test_tensor[row_indices,test_pos] = 0
        
        pos_items = torch.Tensor(items_array[test_pos]).to(device)
        neg_items = torch.Tensor(items_array[test_neg]).to(device)
        
        pos_output = torch.diagonal(model(test_tensor, pos_items).to(device))
        neg_output = torch.diagonal(model(test_tensor, neg_items).to(device))
        
        pos_loss = torch.mean((torch.ones_like(pos_output)-pos_output)**2)
        neg_loss = torch.mean((neg_output)**2)
        print(f'test pos_loss = {pos_loss}, neg_loss = {neg_loss}')
        
        hit_rate_at_10, hit_rate_at_50, hit_rate_at_100, MRR, MPR = recommender_evaluations(model, **kw_dict)
        hr10.append(hit_rate_at_10) # metric for monitoring
        print(f" epoch {epoch} hit_rate_at_10 {hit_rate_at_10} hit_rate_at_50 {hit_rate_at_50} hit_rate_at_100 {hit_rate_at_100} MRR {MRR} MPR {MPR}")
        
        test_losses.append(-hit_rate_at_10)
        if epoch>5: # early stop if the HR@10 decreases for 4 epochs in a row
            if test_losses[-2]<=test_losses[-1] and test_losses[-3]<=test_losses[-2] and test_losses[-4]<=test_losses[-3]:
                logger.info(f'Early stop at trial with batch size = {batch_size} and lr = {lr}. Best results at epoch {np.argmin(test_losses)} with value {np.min(test_losses)}')
                train_losses_dict[trial.number] = train_losses
                test_losses_dict[trial.number] = test_losses
                HR10_dict[trial.number] = hr10
                return max(hr10)
            
    logger.info(f'Stop at trial with batch size = {batch_size} and lr = {lr}. Best results at epoch {np.argmin(test_losses)} with value {np.min(test_losses)}')
    train_losses_dict[trial.number] = train_losses
    test_losses_dict[trial.number] = test_losses
    HR10_dict[trial.number] = hr10
    return max(hr10)

In [58]:
train_losses_dict = {}
test_losses_dict = {}
HR10_dict = {}

VAE_config= {
"enc_dims": [256,64],
"dropout": 0.5,
"anneal_cap": 0.2,
"total_anneal_steps": 200000
}

def VAE_objective(trial):
    
    lr = 0.001
    batch_size = 256
    epochs = 30
    model = VAE(VAE_config ,**kw_dict)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    hr10 = []
    print('======================== new run ========================')
    logger.info('======================== new run ========================')
    
    for epoch in range(epochs):
        if epoch!=0 and epoch%10 == 0:
            lr = 0.1*lr
            optimizer.lr = lr
        loss = model.train_one_epoch(train_array, optimizer, batch_size)
        train_losses.append(loss)
        torch.save(model.state_dict(), Path(checkpoints_path, f'VAE_new_{data_name}_{trial.number}_{epoch}_{round(lr,4)}_{batch_size}.pt'))


        model.eval()
        test_matrix = static_test_data.to_numpy()
        test_tensor = torch.Tensor(test_matrix[:,:-2]).to(device)
        test_pos = test_array[:,-2]
        test_neg = test_array[:,-1]
        row_indices = np.arange(test_matrix.shape[0])
        test_tensor[row_indices,test_pos] = 0
        output = model(test_tensor).to(device)
        pos_loss = -output[row_indices,test_pos].mean()
        neg_loss = output[row_indices,test_neg].mean()
        print(f'pos_loss = {pos_loss}, neg_loss = {neg_loss}')
        
        hit_rate_at_10, hit_rate_at_50, hit_rate_at_100, MRR, MPR = recommender_evaluations(model, **kw_dict)
        hr10.append(hit_rate_at_10)
        print(hit_rate_at_10, hit_rate_at_50, hit_rate_at_100, MRR, MPR)
        
        test_losses.append(pos_loss.item())
        if epoch>5:
            if test_losses[-2]<test_losses[-1] and test_losses[-3]<test_losses[-2] and test_losses[-4]<test_losses[-3]:
                logger.info(f'Early stop at trial with batch size = {batch_size} and lr = {lr}. Best results at epoch {np.argmin(test_losses)} with value {np.min(test_losses)}')
                train_losses_dict[trial.number] = train_losses
                test_losses_dict[trial.number] = test_losses
                HR10_dict[trial.number] = hr10
                return max(hr10)
    
    logger.info(f'Stop at trial with batch size = {batch_size} and lr = {lr}. Best results at epoch {np.argmin(test_losses)} with value {np.min(test_losses)}')
    train_losses_dict[trial.number] = train_losses
    test_losses_dict[trial.number] = test_losses
    HR10_dict[trial.number] = hr10
    return max(hr10)

In [59]:
logger = logging.getLogger()

logger.setLevel(logging.INFO)  # Setup the root logger.
#logger.addHandler(logging.FileHandler(f"{recommender_name}_{data_name}_Optuna.log", mode="w"))

optuna.logging.enable_propagation()  # Propagate logs to the root logger.
optuna.logging.disable_default_handler()  # Stop showing logs in sys.stderr.

study = optuna.create_study(direction='maximize')

logger.info("Start optimization.")

if recommender_name == 'MLP':
    study.optimize(MLP_objective, n_trials=1) 
elif recommender_name == 'VAE':
    study.optimize(VAE_objective, n_trials=20) 

with open(f"{recommender_name}_{data_name}_Optuna.log") as f:
    assert f.readline().startswith("A new study created")
    assert f.readline() == "Start optimization.\n"
    
    
# Print best hyperparameters and corresponding metric value
print("Best hyperparameters: {}".format(study.best_params))
print("Best metric value: {}".format(study.best_value))

======================== new run ========================
(  0 /  77) loss = 445.3829
pos_loss = -4.48880746262148e-05, neg_loss = 4.6564804506488144e-05
0.04221635883905013 0.11508017048914146 0.17860767201136593 0.005074081591231987 21.786734562769432
(  0 /  77) loss = 396.0394
pos_loss = -1.9561757653718814e-05, neg_loss = 2.387848508078605e-05
0.08057641566876396 0.21006697787700426 0.2947026588187538 0.020702252892226505 14.987659438987919
(  0 /  77) loss = 380.7500
pos_loss = -2.624195440148469e-05, neg_loss = 3.0193852580850944e-05
0.10858534605236453 0.24964481428861376 0.33590420133955756 0.02800893038360057 12.738238303532915
(  0 /  77) loss = 356.3014
pos_loss = -2.415739254502114e-05, neg_loss = 2.9212022127467208e-05
0.125634260198904 0.27440633245382584 0.36675461741424803 0.032474122183884714 11.540856776247955
(  0 /  77) loss = 342.6116
pos_loss = -1.4232836292649154e-05, neg_loss = 2.0820378267671913e-05
0.1376090927542115 0.3003856302009336 0.39253095189770654 0.0

Trial 0 failed with parameters: {} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_993892/787139549.py", line 46, in VAE_objective
    hit_rate_at_10, hit_rate_at_50, hit_rate_at_100, MRR, MPR = recommender_evaluations(model, **kw_dict)
  File "/home/mvarasteh/ProvidersExplantion/help_functions.ipynb", line 109, in recommender_evaluations
    "    return dict(sorted_items_by_prob)"
  File "/home/mvarasteh/ProvidersExplantion/help_functions.ipynb", line 54, in get_index_in_the_list
    "    matrix = np.array(data)[:,:num_items] # keep only items columns, remove demographic features columns\n",
  File "/home/mvarasteh/ProvidersExplantion/help_functions.ipynb", line -1, in get_top_k
KeyboardInterrupt
Trial 0 failed with value None.


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3550, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_993892/2875115496.py", line 16, in <module>
    study.optimize(VAE_objective, n_trials=20)
  File "/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-packages/optuna/study/study.py", line 475, in optimize
    _optimize(
  File "/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 63, in _optimize
    _optimize_sequential(
  File "/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 160, in _optimize_sequential
    frozen_trial = _run_trial(study, func, catch)
  File "/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 248, in _run_trial
    raise func_err
  File "/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-pa

In [ ]:
checkpoints_path='/home/mvarasteh/ProvidersExplantion/checkpoints'
recommender_path='MLP_lastfm_0.0001_512_0_16.pt'



In [ ]:
def load_recommender():
    if recommender_name=='MLP':
        recommender = MLP(hidden_dim, **kw_dict)
    elif recommender_name=='VAE':
        recommender = VAE(VAE_config, **kw_dict)

    recommender_checkpoint = torch.load(Path(checkpoints_path, recommender_path))
    recommender.load_state_dict(recommender_checkpoint)
    recommender.eval()
    for param in recommender.parameters():
        param.requires_grad= False
    return recommender
    
recommender = load_recommender()

In [ ]:
train_data['user_id']=train_data.index
test_data['user_id']=test_data.index
train_array=train_data.to_numpy()

#filtered_users=int(train_array.shape[0]*0.1)
#train_array=train_array[0:filtered_users]

test_array=test_data.to_numpy()

#filtered_users=int(test_array.shape[0]*0.1)
#test_array=test_array[0:filtered_users]

all_array=np.concatenate([train_array,test_array])

In [ ]:
all_data=pd.concat((train_data,test_data), axis=0)

In [ ]:
def recommendations_generation(input_array):
    topk_itms = {}

    for i in range(input_array.shape[0]):
        user_index = input_array[i][-1]
        user_tensor = torch.Tensor(input_array[i][:-1]).to(device)
        
        topk_itms[user_index] = get_user_recommended_item(user_tensor, recommender, **kw_dict).cpu().numpy()
    return topk_itms

topk_itms=recommendations_generation(all_array)

In [ ]:
Items_df=Itesm_df

In [ ]:
## Converting each genre to a multihot vector 
ID2genre_hot=pd.concat([Items_df["track_id"], Items_df['genre'].str.get_dummies(sep='|')], axis=1)

In [ ]:
def genre (input_array, item2genre, users, Targ_genre):
    
    '''
    Counting the occurences of each genre in users profiles
    input_array: all users interactions
    item2genre: itmes to genre dataframe(multihot)
    users: all the users 
    Targ_genre: genres of the target item

    '''
    A=input_array[:,:-1]
    if users==[]:
        return 0
    else:
         z=sum( item2genre.set_index('track_id').loc[np.nonzero(A[uid])[0]].sum(axis=0) for uid in users)
         return z[Targ_genre].mean()

In [ ]:
def genre_profile(input_array, item2genre, users, Targ_genre):
  
    '''
    Counting the occurences of each genre in users profiles
    input_array: all users interactions
    item2genre: itmes to genre dataframe(multihot)
    users: all the users 
    Targ_genre: genres of the target item

    '''

    # Ensure track_id is the index so we can look up tracks instantly
    if 'track_id' in item2genre.columns:
        genre_lookup = item2genre.set_index('track_id')
    else:
        genre_lookup = item2genre

    if not users:
        return 0

    # 2. Get the track columns (excluding the last metadata columns like 'pos'/'neg')
    # Assuming input_array is a numpy array of interactions
    A = input_array[:, :-1] 

    total_genre_counts = None

    for uid in users:
        # Get indices where user has a 1 (interaction)
        interacted_track_indices = np.nonzero(A[uid])[0]
        
        if len(interacted_track_indices) == 0:
            continue
            
        # Select the genres for these tracks using .iloc (positional indexing)
        user_genres = genre_lookup.iloc[interacted_track_indices].sum(axis=0)
        
        if total_genre_counts is None:
            total_genre_counts = user_genres
        else:
            total_genre_counts += user_genres

    if total_genre_counts is None:
        return 0

   
    return total_genre_counts[Targ_genre].mean()

In [ ]:
ID_Year=Items_df.set_index('track_id')['year']

In [ ]:
def recency_profiles(input_array, users, ID_Year):
    
    if not users:
        return 0
  
    else: 
        avg_time_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg_time=ID_Year[idx].mean()
            avg_time_all.append(avg_time)
        return np.mean(avg_time_all)
    

In [ ]:
ID_energy=Items_df.set_index('track_id')['energy']

def energy_profiles(input_array, users, ID_energy):
    
    if not users:

        return 0
    
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_energy[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    

    
ID_liveness=Items_df.set_index('track_id')['liveness']

def liveness_profiles(input_array, users, ID_liveness):
    
    if not users:
        return 0  
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_liveness[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    

ID_dance=Items_df.set_index('track_id')['danceability']
ID_instrument=Items_df.set_index('track_id')['instrumentalness']

def dance_profiles(input_array, users, ID_dance):
    
    if not users:

        return 0
    
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_dance[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    
def instrument_profiles(input_array, users, ID_instrument):
    
    if not users:
        return 0

    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_instrument[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)


In [ ]:
sorted_pop=sorted(pop_dict.items(), key=lambda x:x[1], reverse=True)[:100]

Topk_pop, _=list(zip(*sorted_pop))

recomms_list=np.concatenate(list(topk_itms.values()))

num_users=all_array.shape[0]
num_items=all_array.shape[1]

In [ ]:
recommendation_matrix = np.zeros((num_users, num_items), dtype=int)
for user, items in topk_itms.items():

    user=user.astype(int)
    items=items.astype(int)
    recommendation_matrix[user, items] = 1 


In [ ]:
## list of users for each item in recommendations
from collections import defaultdict

item2users = defaultdict(list)

for user, items in topk_itms.items():
    for item in items:
        item2users[int(item)].append(int(user))


In [ ]:
all_values=np.concatenate(list(topk_itms.values()))
#items_indx=np.unique(all_values)
items_indx=np.arange( num_items)
count=Counter(all_values)
count_all={item: count.get(item, 0) for item in items_indx}

In [ ]:
def profile_popularity(input_array, users):
         
      '''
      computing average profiles populairty for users that recived the target item as a reocommendation 
      input_array: alll the interactions for each user and item
      users: all the users that recived the target item as a reocmmendation

      '''
      if not users:

        return 0
      
      else:

        All_pop = [] ## stores all pop scores for each users
    
        for u_id in users:
            ## profile for users
            user_profile=input_array[u_id,0:-1]

            ## get popularity of items in user profile
            mu=sum(pop_dict[int(item)] for item in np.nonzero(user_profile)[0]) / len(np.nonzero(user_profile)[0])
            
            All_pop.append(mu)

        return np.mean(All_pop)



In [ ]:
users_size = {}

for user_id, row in enumerate(all_array):
            user_profile = row[:-1]
            #user_id = row[-1]
            users_size[user_id] = np.count_nonzero(user_profile)

In [ ]:

def profile_size(input_array, users, users_size):

    
    """
    input_array: np.array of shape [num_users, num_items+1]
                 Last column is user_id
    users: list of user_ids who received the target item as recommendation
    
    Returns:
        Average normalized profile size for the users
    """
   
    if not users:

        return 0
    
    else: 
    
        # Count the number of items for each user
        
        
        # Normalize profile sizes
        #counts = np.array(list(items_count.values()))
        #max_val = counts.max()
        #min_val = counts.min()
        
        # Avoid division by zero
        #if max_val == min_val:
            #normalized_counts = {uid: 1.0 for uid in items_count}
        #else:
        #normalized_counts = {uid: (count - min_val) / (max_val - min_val) 
                                #for uid, count in items_count.items()}
        
        # Average over the target users
        #prof_size = np.mean([normalized_counts[u] for u in users if u in normalized_counts])
        return sum(users_size[uid] for uid in users)/len(users)
        #return prof_size


In [ ]:
#cosine_sim_users=cosine_similarity(all_array[:,:-1])
users_embeddings=recommender.users_fc.weight.cpu().numpy()
users_embeddings=(all_array[:,:-1] @ (users_embeddings.T))
cosine_sim=cosine_similarity(users_embeddings)
mask=np.ones_like(cosine_sim)
np.fill_diagonal(mask,0)

similar_users_all = np.argsort(cosine_sim * mask)[:,::-1]

users_id=np.arange(similar_users_all.shape[0])
id_sim_users=similar_users_all[:,:5]
sim_users=dict(zip(users_id, id_sim_users))
df_sim_users=pd.DataFrame(sim_users.items(), index=None, columns=['user_id', 'similar_users'])

In [ ]:
df_sim_users

In [ ]:
mask=np.ones_like(cosine_sim_users)
np.fill_diagonal(mask,0)
x=np.sort(cosine_sim_users * mask)[:,::-1]

In [ ]:
users_indx, items_index = np.where(x>0.8)

In [ ]:
from collections import  defaultdict
sim_users=defaultdict(list)
for u , i in zip(users_indx, items_index):
    sim_users[u].append(i)


In [ ]:

def Targ_in_neighbs_profile (input_array, similar_users, users, targ_item, drop_last_col=True):
    """
    Count, for each user t, how many of t's similar users have targ_item (observed/consumed).
    input_array: 2D array [n_users, n_items( + maybe 1 label col)]
    similar_users: dict[int -> list[int]] of similar-user indices
    targ_item: int (column index in the **effective** item matrix)
    k: optional cap on number of similar users
    drop_last_col: set True if the last column is a label you want to ignore
    """
    if not users:
        return 0
    A = input_array[:, :-1] if drop_last_col else input_array
    # Boolean vector: does user u have targ_item?
    Target_has_item = A[:, targ_item] != 0

    dict_sim_users = dict(similar_users.iloc[users].values)
    counts = {}
    
    for t, S in dict_sim_users.items():
        #counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) - int(Comp_has_item[np.asarray(S, dtype=int)].sum())
        counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) 

    #min_val = min(counts.values())
    #max_val = max(counts.values())
    #range_val = (max_val - min_val)

    #Targ_in_neighb_prof = {k: (v - min_val) / range_val for k, v in counts.items()}
    
    #item2users_neighbprof=sum(counts[u] for u in item2users.get(targ_item,[0]) )/len (item2users.get(targ_item,[0]))
    item2users_neighbprof = sum(counts.values()) / len(counts)

    
    return item2users_neighbprof


In [ ]:

def Targ_in_neighbs_recomm(input_array, similar_users, users, targ_item, drop_last_col=False):

    """
    Count, for each item i, how many of i's similar users have targ_item (observed/consumed).
    input_array: 2D array [n_users, n_items( + maybe 1 label col)]
    similar_users: dict[int -> list[int]] of similar-user indices
    recommendations: dict[int -> list[int]] of recommended items for each user
    targ_item and Comp_item: int (column index in the **effective** item matrix)
    k: optional cap on number of similar users
    drop_last_col: set True if the last column is a label you want to ignore
    """
   

    A = input_array[:, :-1] if drop_last_col else input_array
    # Boolean vector: does user u have targ_item?
    Target_has_item = A[:, targ_item] != 0


    dict_sim_users = dict(similar_users.iloc[users].values)
    counts = {}
    
    for t, S in dict_sim_users.items():
        #counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) - int(Comp_has_item[np.asarray(S, dtype=int)].sum())
        counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) 

    
    #min_val = min(counts.values())
    #max_val = max(counts.values())
    #range_val = (max_val - min_val)
    #print(f"range_val is {range_val}")
    #Targ_in_neighb_recomm = {k: (v - min_val) / range_val for k, v in counts.items()}


    #item2users_neighbrecomms=sum(counts[u] for u in item2users.get(targ_item,[0]) )/len (item2users.get(targ_item,[0]))
    
    item2users_neighbrecomms =  0 if not users else sum(counts.values()) / len(counts)

    
    return item2users_neighbrecomms




In [ ]:
cosine_sim_items=cosine_similarity(all_array[:,:-1].T)
#items_embeddings=np.array(recommender.items_fc.weight.detach())
#cosine_sim=cosine_similarity(items_embeddings.T)
mask=np.ones_like(cosine_sim_items)
np.fill_diagonal(mask,0)
similar_items_all = np.argsort(cosine_sim_items * mask)[:,::-1]
itms_id=np.arange(similar_items_all.shape[0])
id_sim_itms=similar_items_all[:,:10]
sim_itms=dict(zip(itms_id, id_sim_itms))

In [ ]:

def neighbors_item_profile(input_array, similar_items, users, targ_item, drop_last_col=True):
    """
    For each user u, count how many of u's items are in targ_item's similar-items list.
    input_array: 2D array [n_users, n_items (+ maybe 1 user-id col)]
                 Assumes item columns are binary (0/1).
    similar_items: dict[int -> list[int]]  (item -> similar item indices)
    targ_item: int (column index in the effective item matrix)
    drop_last_col: if True, treat the last column of input_array as a user-id (ignored for items)
    """
    A = input_array[:, :-1] if drop_last_col else input_array
    user_ids = input_array[:, -1].astype(int) if drop_last_col else np.arange(A.shape[0])

    # indices of items similar to targ_item
    sim_idx = np.array(similar_items.get(targ_item, []), dtype=int)

    if len(similar_items[targ_item]) == 0:
        return 0

    # Ensure binary (handles 0/1 or counts)
    Ab = (A != 0)

    # Count, per user, how many of their items intersect sim_idx (vectorized slice + sum)
    counts_vec = Ab[:, sim_idx].sum(axis=1)
    count={int(u): int(c) for u, c in zip(user_ids, counts_vec)}

    # Return dict[user_id] -> count
    #min_val = min(count.values())
    #max_val = max(count.values())
    #range_val = (max_val - min_val)

    #Neighb_items_in_prof = {k: (v - min_val) / range_val for k, v in count.items()}
    
    item2users_neighbritms = 0 if not users else sum(count[u] for u in users) / len(users)

    
    return item2users_neighbritms


In [ ]:
all_items=Items_df['track_id'].values

In [ ]:
len(under_expos_itms)

In [ ]:

#df=pd.DataFrame(columns=['targ_id','Targ_itm_pop','Profile_pop_norm' ,'prof_size', 'targ_itm_timestamp_norm','profiles_recency', 
                           #   'Targ_itm_avg_rating','genre', 'targ_in_neighb_prof','Targ_in_neighb_recomm','neighb_itms_in_prof','Exposure'
                          #   ])
df=pd.DataFrame()


num_rows=num_items
idx= torch.randperm(num_rows)[:1000]
#samples=topk_itms[idx]
count_pos=0
count_neg=0


over_expos_itms = np.array(list(item2users.keys()))  # <-- convert set to list first
under_expos_itms = list(set(Items_df['track_id'].unique()) - set(over_expos_itms)) # -1 --. MovieIDs in Ratings_df starts from 1

# Sample safely
sample_size = 3000
sampled_items = np.random.choice(under_expos_itms, size=sample_size, replace=False)

# Concatenate 1D arrays
all_items = np.concatenate([sampled_items, over_expos_itms])

for i in all_items:
#for i in over_expos_itms:
        i=int(i)
        item_users=item2users.get(i,[])
        itm_genre=Items_df.loc[Items_df['track_id']==i,'genre' ].str.split('|').iloc[0]
        row = {
                   
                    'TargID': i,
                    'Items Popularity':pop_dict[i] ,
                    
                    'Profile Size':profile_size(all_array, item_users, users_size),
                     'Profiles Time':recency_profiles(all_array,item_users, ID_Year),
                    'Profiles Popularity': profile_popularity(all_array, item_users),
                    'Items Time':ID_Year[i],
                    'Genre Affinity':genre(all_array,ID2genre_hot, item_users,itm_genre ),
                     'targ_energy':ID_energy[i],
                    'energy_level_profile': energy_profiles(all_array, item_users, ID_energy),
                    'liveness_profile': liveness_profiles(all_array, item_users, ID_liveness),
                    'liveness':ID_liveness[i],
                    'profiles_dance': dance_profiles(all_array, item_users, ID_dance),
                    'target_dance':ID_dance[i],
                    #'instrument_profile': instruemnt_profiles(all_array, item_users, ID_instrument),
                    'target_instrument':ID_instrument[i],

                     #'influence_user' : influ_users_impact(all_array, i, infl_users ),
                    'Neighbors Profiles': Targ_in_neighbs_profile (all_array, df_sim_users, item_users, i, drop_last_col=True),
                    'Neighbors Recommendations': Targ_in_neighbs_recomm(recommendation_matrix, df_sim_users, item_users, i, drop_last_col=False),
                    #'Similar Items Profile': neighbors_item_profile(all_array, sim_itms, item_users, i, drop_last_col=True),

                    'Exposure': count.get(i, 0)
                                    }
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        #indx=(torch.randperm(49)+1)[0:5]

   
        


In [ ]:
df

In [ ]:
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split 
## Get project root (one level up from notebooks/)
#project_root = Path.cwd().parent
#sys.path.append(str(project_root))
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

In [ ]:

scaler=MinMaxScaler()
array_scaled=scaler.fit_transform(df)

df_scaled=pd.DataFrame(array_scaled, columns=df.columns)
X=df_scaled.drop(columns=['Exposure','TargID',])
y=df_scaled['Exposure']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

In [ ]:
y

In [ ]:
model=LinearRegression()
model.fit(X_train, y_train)
print(f" The R^2 score is {model.score(X_test,y_test)}")
list(zip(X_test.columns, model.coef_))

y_hat=model.predict(X_test)
residuals=y_hat-y_test
plt.scatter(y_hat,residuals )
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('y_pred')
plt.ylabel('Residuals')
plt.title(f"recommender model: {recommender_name}")
plt.show()

In [ ]:
dict(zip(X.columns, model.coef_))

In [ ]:
import matplotlib.pyplot as plt

features=X.columns.values

n_cols = 3
n_rows = (len(features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 10))
axes = axes.flatten()

for ax, feat in zip(axes, features):
    df_scaled.plot.scatter(x=feat, y='Exposure', ax=ax)
    ax.set_title(f' Exposure vs {feat}')

# Remove empty plots
for i in range(len(features), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()

plt.show()


In [ ]:
df.corr()['Exposure'].sort_values(ascending=False)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regr = RandomForestRegressor(max_depth=7, random_state=44)
regr.fit(X_train, y_train)
print(f"The R2 score is {regr.score(X_test, y_test)}")
print(f"Features Importance {dict(zip(X.columns, regr.feature_importances_))}")
y_hat=regr.predict(X_test)

residuals=y_hat-y_test
plt.scatter(y_hat,residuals )
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('y_pred')
plt.ylabel('Residuals')
plt.title(f"recommender model: {recommender_name}")
plt.show()


In [ ]:
regr.score(X_test, y_test)

In [ ]:
id_exp=pd.DataFrame(count.items(), columns=['track_id', 'Exposure'])

In [ ]:
Items_df_tmp=copy.deepcopy(Items_df)

In [ ]:
Items_df_tmp=Items_df_tmp.set_index('track_id')

In [ ]:
Items_df_tmp['Exposure']=Items_df_tmp.loc[id_exp]

In [ ]:
count

In [ ]:
Items_df['exposure']=Items_df['track_id'].map(count)

In [ ]:
Items_df.select_dtypes(exclude=['object', 'string']).corr()['exposure']

In [ ]:
Items_df['tags'].unique().size

In [ ]:
Items_df['genre'].unique()

In [ ]:
Items_df